# 06 · Độ nhạy với dữ liệu thiếu và nhất quán giữa nhóm bệnh nhân (phụ trách: Chiến · Chưởng)

**Đề:** *"missing-data sensitivity"*, *"Assess performance consistency across relevant patient subgroups"*.
Mã nguồn: `trustworthy_ai/robustness.py`, `trustworthy_ai/subgroup.py`.

In [ ]:
# --- Thiết lập chung ---------------------------------------------------------------
import json, os, sys
from pathlib import Path
import pandas as pd, numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")

REAL = False          # True trên EC2 sau khi đã chạy `make trust TASK=...` với dữ liệu thật
TASK = "p1"           # p1 | p2 | p3

def load_results(task=TASK):
    p = ROOT / ("results/trust" if REAL else "artifacts/demo_results") / task / "results.json"
    if not p.exists():
        raise FileNotFoundError(f"Chưa có {p}. Chạy `python tools/make_demo_bundles.py` (demo) hoặc `make trust TASK={task}` (EC2).")
    return json.loads(p.read_text(encoding="utf-8"))

VERIFIED = json.loads((ROOT / "results/data/verified_v1.json").read_text(encoding="utf-8"))
print("Nguồn:", "DỮ LIỆU THẬT" if REAL else "DEMO (dữ liệu giả — số liệu không có ý nghĩa y khoa)")

In [ ]:
R = load_results()

## 1. Che dần thông tin: PR-AUC và O/E giảm thế nào

In [ ]:
rob = pd.DataFrame(R.get('robustness_test') or R['robustness_dev']); rob.pivot_table(index='scenario', columns='rate', values='pr_auc')

In [ ]:
rob.pivot_table(index='scenario', columns='rate', values='oe_ratio')

## 2. Hiệu năng theo nhóm (có CI) và chênh lệch công bằng

In [ ]:
sg = pd.DataFrame(R.get('subgroup_test') or R['subgroup_dev']); sg[['attribute','group','n','n_pos','roc_auc','pr_auc_lift','oe_ratio','recall']]

In [ ]:
pd.DataFrame(R.get('fairness_test') or R['fairness_dev'])

In [ ]:
pd.Series(R['subgroup_relevance']).to_frame('vì sao relevant')

## 3. Phân tích độ nhạy của thiết kế (C0–C6)

In [ ]:
pd.DataFrame(R['sensitivity'])

## Câu hỏi tự kiểm tra
1. Vì sao so PR-AUC giữa các nhóm phải dùng lift?
2. Nhóm nào có O/E xa 1 nhất? Điều đó nguy hiểm thế nào?
3. Kịch bản thiếu nào làm mô hình giảm nhanh nhất? Liên hệ với luật gắn cờ HIGH_MISSINGNESS.
4. Equal opportunity (recall gap) là gì?